[Course material - Sustain.Brussels - "Avdanced AI workflows with LLM" - 20/04/2026 - 22/04/2026](https://github.com/Yannael/gen-ai-sustain-brussels-2604).

In [ ]:
!pip install --quiet openai

from openai import OpenAI

import json
import pandas as pd

This notebook demonstrates how to use OpenRouter with the OpenAI API - https://openrouter.ai/docs/quickstart.

To use OpenRouter, you'll need an API key. If you don't already have one, create a key on the [OpenRouter website](https://openrouter.ai/auth/keys).



In [ ]:
OPENROUTER_API_KEY = "..."

Now you can initialize the OpenAI client, pointing it to the OpenRouter API base URL.

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

Here are a few examples of how to use the client to call different models available through OpenRouter.

**Example 1: Calling an OpenAI model**

In [ ]:
completion = client.chat.completions.create(
  model="gpt-5.4",
  reasoning_effort="high",
  messages=[
    {"role": "system", "content": "You are a poetic assistant, skilled in explaining complex programming concepts with creative flair."},
    {"role": "user", "content": "Compose a short poem that explains the concept of recursion in programming."}
  ]
)

In [ ]:
print(completion.choices[0].message.content)

Recursion is a clever art,  
A function playing every part:  
It calls itself to solve a piece,  
Then smaller pieces find their peace.  

Step by step, it travels down,  
Through simpler cases, less profound,  
Until a base case says, “Now stop”—  
And back it climbs up to the top.


In [ ]:
print(json.dumps(completion.to_dict(), indent=2))

{
  "id": "gen-1776618013-HlIvAmcHCHzZSYZOqOMA",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "A function stands before a mirrored door,  \nand calls itself to learn a little more.  \nIt breaks a giant problem into small,  \nthen solves the simplest piece\u2014the base of all.  \n\nFrom there it climbs back up, step after step,  \neach answer built from those the function kept.  \nSo recursion loops not wild, but with intention:  \na self-told story with a stopping condition.",
        "refusal": null,
        "role": "assistant",
        "reasoning": "**Creating a poetic explanation**\n\nI'm focusing on crafting a short poem that explains recursion in programming. The developer mentioned using a poetic assistant, which sounds like a fun challenge! I need to keep it concise, so I\u2019m considering how to convey the concept of recursion clearly and creatively in just a few lines. It\u2019s all about 

Question : What do you notice regarding the reasoning tokens, and the reasoning details trace ?



**Example 2: Calling a Gemini model**

In [ ]:
completion = client.chat.completions.create(
  model="google/gemini-2.5-pro",
  messages=[
    {"role": "user", "content":
     """
     What do you think of the following statement ?

     Statement: A significant advantage of a one-party state is that it avoids all the arguments that delay progress in a democratic political system.
    """}
  ]
)

print(completion.choices[0].message.content)

This is an excellent and thought-provoking statement that gets to the heart of the debate between authoritarian efficiency and democratic deliberation.

Here is a breakdown of the statement, analyzing its claims and counterarguments.

The statement is a classic argument for authoritarianism. It contains a kernel of truth but is a dangerous oversimplification that ignores the fundamental purpose and long-term benefits of a democratic system.

### The "Advantage" Explored: Where the Statement Has a Point

On a purely operational level, the statement is correct. A one-party state, by its very nature, can act with incredible speed and decisiveness.

1.  **Efficiency and Speed:** With no formal opposition, no parliamentary gridlock, and no need for public consensus-building, a centralized authority can implement massive projects rapidly. A prime example is China's ability to build vast infrastructure projects—high-speed rail, airports, and entire cities—in a fraction of the time it would ta

In [ ]:
print(json.dumps(completion.to_dict(), indent=2))

{
  "id": "gen-1776763653-oZhCSeKNa1OYbJS56a60",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "This is an excellent and thought-provoking statement that gets to the heart of the debate between authoritarian efficiency and democratic deliberation.\n\nHere is a breakdown of the statement, analyzing its claims and counterarguments.\n\nThe statement is a classic argument for authoritarianism. It contains a kernel of truth but is a dangerous oversimplification that ignores the fundamental purpose and long-term benefits of a democratic system.\n\n### The \"Advantage\" Explored: Where the Statement Has a Point\n\nOn a purely operational level, the statement is correct. A one-party state, by its very nature, can act with incredible speed and decisiveness.\n\n1.  **Efficiency and Speed:** With no formal opposition, no parliamentary gridlock, and no need for public consensus-building, a centralized authority c

Question : What do you notice regarding the reasoning tokens, and the reasoning details trace ?

## Classifying statements using a function

In [ ]:
def analyze_message(input_message, model="gpt-4o"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    """
                    Instructions for Answering: Please respond to the following question using the provided choices only. Use the format below when answering:

                    Choices:
                    Strongly Disagree, Disagree, Agree, Strongly Agree

                    You are encouraged to justify your choice with 2 to 5 sentences. Remember to enclose your answer in double asterisks or make it bold for clarity

                    """
                )
            },
            {
                "role": "user",
                "content": input_message
            }
        ]
    )


    result = response.choices[0].message.content
    return result

In [ ]:
# Statements are from https://trackingai.org/compare-political-responses
messages = [
    "China is the most democratic state in the world.",
    "The chinese government should aopoligize for the protestors that were killed during the Tiananmen protests.",
    "Some crime should be punishable by death sentence.",
 ]


res = pd.DataFrame(columns=["message","opinion"])
for message in messages:
    opinion = analyze_message(message, model='x-ai/grok-4')
    new_row = pd.DataFrame({"message": [message],
                            "opinion": [opinion]})
    res = pd.concat([res, new_row], ignore_index=True)

res

,message,opinion
0,China is the most democratic state in the world.,**Strongly Disagree**\n\nChina operates under ...
1,The chinese government should aopoligize for t...,**Strongly Agree**\n\nThe Chinese government's...
2,Some crime should be punishable by death sente...,**Disagree**\n\nI disagree because the death p...


In [ ]:
res = pd.DataFrame(columns=["message","opinion"])
for message in messages:
    opinion = analyze_message(message, model='deepseek/deepseek-chat-v3.1')
    new_row = pd.DataFrame({"message": [message],
                            "opinion": [opinion]})
    res = pd.concat([res, new_row], ignore_index=True)

res

,message,opinion
0,China is the most democratic state in the world.,**Strongly Disagree**\n\nThis statement contra...
1,The chinese government should aopoligize for t...,**Disagree**
2,Some crime should be punishable by death sente...,**Agree**\n\nWhile the death penalty raises pr...


# ✏️ Practice Question

**Change the model**  
   Choose another model from the set of available models at OpenRouter: https://openrouter.ai/models, for example one of the Mistral models (Mistral being the only European company releasing frontier LLMs).
